In [118]:
import cv2
import numpy as np
import os

N = 9

In [119]:
def divide_cell(img_dir="board.png", margin = 0.1):
    out_dir = "cells"
    os.makedirs(out_dir, exist_ok=True)
    img = cv2.imread(img_dir, cv2.IMREAD_GRAYSCALE)
    _, img = cv2.threshold(img, 128, 255, cv2.THRESH_BINARY)
    a,l = img.shape
    ys = np.round(np.linspace(0, a, 10)).astype(int)
    xs = np.round(np.linspace(0, l, 10)).astype(int)

    for r in range(9):
        for c in range(9):
            y1,y2 = ys[int(r)], ys[int(r+1)]
            x1,x2 = xs[int(c)], xs[int(c+1)]
            cella = img[y1:y2,x1:x2]

            m = int((cella.shape)[0]*margin)
            final_cell = cella[m:-m,m:-m]

            cv2.imwrite(f"{out_dir}/r{r}_c{c}.png", final_cell)

divide_cell()

    

In [120]:
img = cv2.imread(img_dir)
pixels = img.flatten()

In [121]:
img_dir = "cells/r6_c8.png"

def count_black(img_dir):
    img = cv2.imread(img_dir)
    pixels = img.flatten()
    dic = {}
    check = 0
    for k in range(int(len(pixels)/3)):
        if tuple(pixels[(k*3):(k*3)+3]) in dic:
                dic[tuple(pixels[(k*3):(k*3)+3])] = dic[tuple(pixels[(k*3):(k*3)+3])]+1
        else:
            dic[tuple(pixels[(k*3):(k*3)+3])] = 1

    return dic


# preprocess per template
def preprocess(path, out_size=28):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

    # binarizza: cifra bianca su nero
    _, bw = cv2.threshold(img, 0, 255,
                          cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    h, w = bw.shape

    # rimuovi bordi (griglia)
    pad = int(min(h, w) * 0.08)
    bw[:pad, :] = 0
    bw[-pad:, :] = 0
    bw[:, :pad] = 0
    bw[:, -pad:] = 0

    # trova contorni
    contours, _ = cv2.findContours(bw,
                                   cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return None  # cella vuota

    cnt = max(contours, key=cv2.contourArea)

    # scarta roba troppo piccola
    if cv2.contourArea(cnt) < 0.02 * (h*w):
        return None

    x, y, ww, hh = cv2.boundingRect(cnt)
    digit = bw[y:y+hh, x:x+ww]

    # metti in quadrato
    s = max(ww, hh)
    canvas = np.zeros((s, s), dtype=np.uint8)

    y0 = (s - hh) // 2
    x0 = (s - ww) // 2
    canvas[y0:y0+hh, x0:x0+ww] = digit

    # resize finale
    resized = cv2.resize(canvas, (out_size, out_size),
                         interpolation=cv2.INTER_AREA)

    return resized

def load_templates(folder="templates"):
    templates = {}
    for d in range(1, 10):  # niente 0
        path = os.path.join(folder, f"{d}.png")
        templates[d] = preprocess(path)
    return templates

def match_digit(cell_img, templates, min_score=0.2):
    cell = preprocess(cell_img)

    # se quasi vuota → None
    if cv2.countNonZero(cell) < 10:
        return None

    best_digit = None
    best_score = -1

    cell_f = cell.astype(np.float32)/255.0
    cell_f = (cell_f - cell_f.mean()) / (cell_f.std() + 1e-6)

    for d, tmpl in templates.items():
        t = tmpl.astype(np.float32)/255.0
        t = (t - t.mean()) / (t.std() + 1e-6)

        score = float((cell_f * t).mean())

        if score > best_score:
            best_score = score
            best_digit = d

    if best_score < min_score:
        return None

    return best_digit

In [122]:
templates = load_templates()

In [123]:
grid = []

for r in range(N):
    grid.append([])
    for c in range(N):
        img_dir = f"cells/r{r}_c{c}.png"
        best_match = match_digit(img_dir, templates)
        grid[r].append(best_match)

grid

[[1, None, None, None, 3, 4, None, None, 8],
 [None, 7, None, 6, 8, None, None, 3, None],
 [None, None, 8, 2, 1, None, 7, None, 4],
 [None, 5, 4, None, 9, None, 6, 8, None],
 [9, 1, None, 5, None, 8, None, 2, None],
 [None, 8, None, 3, None, None, None, None, 5],
 [3, None, 5, 9, None, 6, 8, 7, 1],
 [None, None, 6, None, None, None, None, 4, None],
 [None, None, 1, None, 7, None, 2, None, None]]